# Chapter 4: Building Models with Distance Metrics and Nearest Neighbors

**Buku:** scikit-learn Cookbook (O'Reilly)
**Topik:** Model berbasis jarak dan k-Nearest Neighbors (KNN)

## Ringkasan Chapter

Manusia mudah membedakan objek yang mirip dan tidak mirip. Mesin melakukannya dengan **mengukur jarak** antar data di ruang fitur. Chapter ini membahas:

1. **Metrik jarak:** Euclidean, Manhattan, Minkowski, Chebyshev.
2. **KNN:** klasifikasi dengan "voting" dari $k$ tetangga terdekat.
3. **Batas keputusan** dan pengaruh nilai $k$ (bias-variance).
4. **Pengaruh metrik jarak** pada kinerja KNN.
5. **Tuning hyperparameter** (`n_neighbors`, `weights`, `metric`) dengan `GridSearchCV`.
6. **Evaluasi model:** cross-validation, learning curve, confusion matrix, classification report.

> Cara pakai: jalankan sel dari atas ke bawah. Setiap sel kode diikuti penjelasan teori dan cara membaca hasilnya.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import warnings
warnings.filterwarnings("ignore")

np.random.seed(2024)
pd.set_option("display.width", 120)

print("scikit-learn:", sklearn.__version__)
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)

**Penjelasan:** sel ini memuat library dasar dan mengunci biji acak (`seed`) agar hasil bisa direproduksi.

## 1. Metrik Jarak

Jarak adalah ukuran seberapa berbeda dua titik $\mathbf{a}$ dan $\mathbf{b}$ di ruang berdimensi $n$.

| Metrik | Rumus | Intuisi |
|:--|:--|:--|
| **Euclidean** | $d = \sqrt{\sum_i (a_i - b_i)^2}$ | Garis lurus (jarak "burung terbang") |
| **Manhattan** | $d = \sum_i \lvert a_i - b_i \rvert$ | Berjalan mengikuti kotak jalan kota |
| **Minkowski** | $d = \left(\sum_i \lvert a_i - b_i \rvert^p\right)^{1/p}$ | Generalisasi: $p=1$ Manhattan, $p=2$ Euclidean |
| **Chebyshev** | $d = \max_i \lvert a_i - b_i \rvert$ | Batas dari Minkowski saat $p \to \infty$ |

Kita hitung untuk titik $A=(1,1)$ dan $B=(4,5)$, lalu bandingkan dengan perhitungan manual.

In [ ]:
from sklearn.metrics import pairwise_distances
from scipy.spatial import distance

A = np.array([1, 1])
B = np.array([4, 5])

# Perhitungan manual
euc_manual = np.sqrt(((A - B) ** 2).sum())
man_manual = np.abs(A - B).sum()

print(f"Euclidean  manual: {euc_manual:.4f} | scipy: {distance.euclidean(A, B):.4f}")
print(f"Manhattan  manual: {man_manual:.4f} | scipy: {distance.cityblock(A, B):.4f}")
for p in [1, 2, 3, 10]:
    print(f"Minkowski p={p:<2}: {distance.minkowski(A, B, p):.4f}")
print(f"Chebyshev        : {distance.chebyshev(A, B):.4f}")

# Matriks jarak antar beberapa titik
P = np.array([[1, 1], [4, 5], [0, 3]])
print("\nMatriks jarak Euclidean antar 3 titik:")
print(pairwise_distances(P, metric="euclidean").round(3))

**Penjelasan hasil:** Euclidean = $\sqrt{3^2+4^2} = 5$, Manhattan = $3+4 = 7$. Minkowski dengan $p=1$ sama dengan Manhattan, $p=2$ sama dengan Euclidean, dan makin besar $p$ nilainya makin turun mendekati Chebyshev (yaitu 4, selisih terbesar antar koordinat). Urutan selalu: Manhattan $\geq$ Euclidean $\geq$ Chebyshev. Matriks jarak bersifat simetris dengan diagonal 0.

### Bentuk "lingkaran" tiap metrik

Himpunan semua titik yang berjarak 1 dari titik asal berbentuk berbeda untuk setiap metrik. Bentuk ini menunjukkan bagaimana tiap metrik memandang "kedekatan".

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
grid = np.linspace(-1.5, 1.5, 400)
xx, yy = np.meshgrid(grid, grid)
pts = np.column_stack([xx.ravel(), yy.ravel()])

for p, color, name in [(1, "tab:green", "Manhattan (p=1)"),
                       (2, "tab:red", "Euclidean (p=2)"),
                       (np.inf, "tab:blue", "Chebyshev (p=inf)")]:
    d = np.linalg.norm(pts, ord=p, axis=1).reshape(xx.shape)
    ax.contour(xx, yy, d, levels=[1], colors=color)
    ax.plot([], [], color=color, label=name)

ax.set_aspect("equal"); ax.legend(loc="upper right")
ax.set_title("Himpunan titik berjarak 1 dari pusat")
ax.grid(alpha=0.3)
plt.show()

**Penjelasan hasil:** Manhattan membentuk belah ketupat (berlian), Euclidean membentuk lingkaran, dan Chebyshev membentuk persegi. Ini memperlihatkan bahwa pemilihan metrik mengubah definisi "tetangga": titik yang dianggap dekat oleh satu metrik bisa dianggap jauh oleh metrik lain.

## 2. Memahami KNN

**k-Nearest Neighbors** memprediksi label titik baru $\mathbf{x}_q$ dengan langkah:

1. Hitung jarak $d(\mathbf{x}_q, \mathbf{x}_i)$ ke semua titik latih.
2. Pilih $k$ titik latih dengan jarak terkecil.
3. **Klasifikasi:** ambil label mayoritas:  $\hat{y} = \text{mode}\{y_i : \mathbf{x}_i \in N_k(\mathbf{x}_q)\}$
4. **Regresi:** ambil rata-rata nilai target tetangga:  $\hat{y} = \dfrac{1}{k}\sum_{\mathbf{x}_i \in N_k} y_i$

KNN disebut **lazy learner**: tahap `fit()` hanya menyimpan data, sedangkan seluruh komputasi terjadi saat `predict()`.

### 2.1 Dataset Iris

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024, stratify=y
)

print(f"Data     : {X.shape[0]} sampel, {X.shape[1]} fitur")
print("Kelas    :", [str(n) for n in iris.target_names])
print("Per kelas:", np.bincount(y))
print("Fitur    :", iris.feature_names)
print(f"Latih: {len(X_train)} | Uji: {len(X_test)}")
print("Proporsi kelas data uji:", np.bincount(y_test))

**Penjelasan hasil:** Iris punya 150 sampel seimbang (50 per kelas) dengan 4 fitur kontinu. Kita pakai `stratify=y` agar proporsi kelas di data latih dan uji sama (10 sampel per kelas pada data uji). Dengan hanya 4 fitur, masalah *curse of dimensionality* belum muncul sehingga metode berbasis jarak seharusnya bekerja baik.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for i, (c, m) in enumerate(zip(["red", "green", "blue"], ["o", "^", "D"])):
    ax.scatter(X[y == i, 2], X[y == i, 3], c=c, marker=m, edgecolor="black",
               alpha=0.7, s=60, label=iris.target_names[i])
ax.set_xlabel("Petal length (cm)"); ax.set_ylabel("Petal width (cm)")
ax.set_title("Iris - fitur petal"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

**Penjelasan hasil:** pada fitur petal, kelas *setosa* membentuk klaster terpisah jelas, sedangkan *versicolor* dan *virginica* berdekatan dengan sedikit tumpang tindih. Artinya KNN akan mudah mengenali setosa, dan kesalahan hampir pasti terjadi di perbatasan versicolor-virginica.

### 2.2 KNN dengan scikit-learn

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print(f"Akurasi data uji : {accuracy_score(y_test, y_pred):.4f}")
print(f"Salah klasifikasi: {(y_pred != y_test).sum()} dari {len(y_test)}")

# Melihat tetangga dari satu sampel uji
dist, idx = knn.kneighbors(X_test[:1])
print("\nSampel uji pertama:", X_test[0])
print("Indeks 3 tetangga  :", idx[0])
print("Jarak ke tetangga  :", dist[0].round(3))
print("Label tetangga     :", y_train[idx[0]], "-> prediksi:", knn.predict(X_test[:1])[0], "| asli:", y_test[0])
print("Probabilitas kelas :", knn.predict_proba(X_test[:1])[0].round(3))

**Penjelasan hasil:** `fit()` hanya menyimpan data latih. Method `kneighbors()` memperlihatkan 3 tetangga terdekat beserta jaraknya, dan prediksi diambil dari label mayoritas mereka. `predict_proba()` pada KNN hanyalah proporsi tetangga per kelas (misalnya 3 tetangga semua kelas 1 menghasilkan probabilitas 1.0).

### 2.3 KNN dari nol (untuk membuktikan cara kerjanya)

Kita tulis ulang KNN dengan NumPy murni lalu bandingkan dengan scikit-learn.

In [ ]:
from collections import Counter

def knn_predict_manual(X_tr, y_tr, X_q, k=3):
    preds = []
    for q in X_q:
        d = np.sqrt(((X_tr - q) ** 2).sum(axis=1))      # jarak Euclidean ke semua titik latih
        nearest = np.argsort(d)[:k]                      # indeks k terdekat
        preds.append(Counter(y_tr[nearest]).most_common(1)[0][0])   # voting mayoritas
    return np.array(preds)

y_manual = knn_predict_manual(X_train, y_train, X_test, k=3)
print("Prediksi manual identik dengan scikit-learn?", np.array_equal(y_manual, y_pred))
print(f"Akurasi manual: {accuracy_score(y_test, y_manual):.4f}")

**Penjelasan hasil:** implementasi manual hanya 5 baris inti (hitung jarak, urutkan, ambil $k$ terdekat, voting) dan menghasilkan prediksi yang sama dengan `KNeighborsClassifier`. Ini membuktikan tidak ada "sihir" di balik KNN. Biayanya: tiap prediksi harus menghitung jarak ke semua titik latih, yaitu $O(n \cdot d)$ per query. Scikit-learn mempercepatnya dengan struktur data seperti KD-tree atau Ball-tree.

### 2.4 Pentingnya scaling pada KNN

Karena KNN murni bergantung pada jarak, fitur berskala besar akan mendominasi. Kita buktikan dengan data Wine.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

Xw, yw = load_wine(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2024)

raw = cross_val_score(KNeighborsClassifier(n_neighbors=5), Xw, yw, cv=cv)
scaled = cross_val_score(
    Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))]),
    Xw, yw, cv=cv)

print(f"Wine, KNN tanpa scaling : {raw.mean():.4f} +/- {raw.std():.4f}")
print(f"Wine, KNN dengan scaling: {scaled.mean():.4f} +/- {scaled.std():.4f}")

**Penjelasan hasil:** pada Wine, fitur `proline` bernilai ratusan sampai ribuan sedangkan fitur lain hanya satu digit, sehingga jarak hampir seluruhnya ditentukan `proline`. Setelah `StandardScaler`, akurasi melonjak. **Aturan praktis:** selalu skalakan fitur sebelum KNN, dan letakkan scaler di dalam `Pipeline` agar tidak terjadi data leakage saat cross-validation. (Iris tidak memerlukannya karena keempat fiturnya sama-sama berskala sentimeter.)

## 3. Batas Keputusan dan Pengaruh Nilai $k$

Nilai $k$ mengatur trade-off bias-variance:

- **$k$ kecil:** batas keputusan rumit dan bergerigi, mengikuti noise (*varians tinggi, bias rendah*, rawan overfitting).
- **$k$ besar:** batas lebih halus dan stabil, tetapi bisa melewatkan pola lokal (*bias tinggi, varians rendah*, rawan underfitting).

Kita visualisasikan memakai dua fitur petal.

In [ ]:
from matplotlib.colors import ListedColormap

X2 = X[:, 2:4]
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=0.2, random_state=2024, stratify=y)

cmap_light = ListedColormap(["#FFAAAA", "#AAFFAA", "#AAAAFF"])
cmap_bold = ListedColormap(["#FF0000", "#00CC00", "#0000FF"])

xmin, xmax = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
ymin, ymax = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(xmin, xmax, 0.02), np.arange(ymin, ymax, 0.02))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, k in zip(axes, [1, 5, 15]):
    m = KNeighborsClassifier(n_neighbors=k).fit(X2_tr, y2_tr)
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.pcolormesh(xx, yy, Z, cmap=cmap_light, shading="auto")
    ax.scatter(X2_tr[:, 0], X2_tr[:, 1], c=y2_tr, cmap=cmap_bold, edgecolor="black", s=30)
    ax.set_title(f"k = {k} | akurasi latih {m.score(X2_tr, y2_tr):.3f} | uji {m.score(X2_te, y2_te):.3f}")
    ax.set_xlabel("Petal length"); ax.set_ylabel("Petal width")
plt.tight_layout()
plt.show()

**Penjelasan hasil:** pada $k=1$ batas bergerigi dan akurasi latih mencapai (atau hampir) 100% karena setiap titik latih menjadi tetangga terdekat bagi dirinya sendiri, ini tanda menghafal data. Pada $k=5$ batas lebih halus, dan pada $k=15$ sangat halus. Perhatikan judul tiap panel: selisih akurasi latih dan uji memberi gambaran overfitting. Pada data sebersih Iris, perbedaan akurasi uji antar $k$ kecil, tetapi pada data berisik $k$ yang terlalu kecil akan jauh lebih merugikan.

## 4. Pengaruh Metrik Jarak

Metrik yang cocok bergantung pada bentuk data. Kita uji pada dua dataset sintetis:

- **Noisy circles:** dua cincin konsentris berisik, batas kelas bersifat radial.
- **Checkerboard:** pola papan catur dengan batas sejajar sumbu koordinat.

In [ ]:
from sklearn.datasets import make_circles

# Dataset 1: lingkaran berisik
X_c, y_c = make_circles(n_samples=300, noise=0.3, factor=0.3, random_state=2024)

# Dataset 2: papan catur 17 x 17
g = np.linspace(0, 4, 17)
gx, gy = np.meshgrid(g, g)
X_b = np.column_stack([gx.ravel(), gy.ravel()])
y_b = np.mod(np.floor(gx.ravel()) + np.floor(gy.ravel()), 2).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_c[:, 0], X_c[:, 1], c=y_c, cmap="coolwarm", edgecolor="black", s=25)
axes[0].set_title(f"Noisy circles (n={len(X_c)})")
axes[1].scatter(X_b[:, 0], X_b[:, 1], c=y_b, cmap="coolwarm", edgecolor="black", s=25)
axes[1].set_title(f"Checkerboard (n={len(X_b)})")
plt.show()

**Penjelasan hasil:** panel kiri memperlihatkan dua cincin yang sangat bercampur akibat noise 0.3. Panel kanan memperlihatkan pola papan catur dengan kotak-kotak berlabel bergantian.

Membandingkan metrik dengan **satu** kali pembagian train/test pada data sekecil ini kurang andal: selisih satu atau dua sampel saja bisa mengubah kesimpulan. Karena itu kita pakai **cross-validation berulang** (5-fold diulang 10 kali).

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=2024)
metrics = ["euclidean", "manhattan", "chebyshev"]

rows = []
for name, (Xd, yd) in {"Noisy circles": (X_c, y_c), "Checkerboard": (X_b, y_b)}.items():
    for metric in metrics:
        s = cross_val_score(KNeighborsClassifier(n_neighbors=3, metric=metric), Xd, yd, cv=rcv)
        rows.append({"Dataset": name, "Metrik": metric, "Akurasi": s.mean(), "Std": s.std()})

res = pd.DataFrame(rows).pivot(index="Metrik", columns="Dataset", values="Akurasi")
print("Akurasi rata-rata (5-fold x 10 ulangan):")
print(res.round(4).to_string())

res.plot(kind="bar", figsize=(8, 5), rot=0, edgecolor="black")
plt.ylabel("Akurasi CV"); plt.title("Pengaruh metrik jarak pada KNN (k=3)")
plt.ylim(0.5, 1.0); plt.legend(loc="lower right")
plt.show()

**Penjelasan hasil:** bandingkan nilai antar baris pada tiap kolom. Pada *noisy circles* ketiga metrik hampir sama (selisihnya jauh di bawah satu poin persen), sehingga tidak ada pemenang yang berarti. Pada *checkerboard*, Euclidean dan Manhattan hampir identik, sedangkan Chebyshev tertinggal jelas. Perhatikan juga bahwa bila metrik dibandingkan lewat satu kali split train/test, selisih sekecil itu bisa tampak seperti kemenangan satu metrik padahal hanya kebetulan. Hasil bisa berubah menurut dataset dan nilai $k$. Pelajaran yang aman ditarik: **tidak ada metrik yang selalu terbaik**. Euclidean adalah default yang masuk akal untuk fitur kontinu berskala sama, sedangkan Manhattan lebih tahan terhadap outlier dan sering dipilih pada data berdimensi tinggi atau data dengan fitur sejajar sumbu. Metrik sebaiknya diperlakukan sebagai hyperparameter dan diputuskan lewat cross-validation, bukan asumsi.

Catatan: `metric="minkowski"` dengan $p=2$ (default) identik dengan Euclidean, sehingga tidak perlu dibandingkan terpisah.

## 5. Tuning Hyperparameter KNN

Hyperparameter utama KNN:

| Hyperparameter | Arti |
|:--|:--|
| `n_neighbors` ($k$) | Jumlah tetangga yang ikut memilih |
| `weights` | `uniform` (semua suara sama) atau `distance` (tetangga lebih dekat berbobot lebih besar, $w_i = 1/d_i$) |
| `metric` | Fungsi jarak yang dipakai |

Kita gunakan `GridSearchCV` dengan scaler **di dalam pipeline**.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier())])

param_grid = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 11, 15],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

print("Parameter terbaik   :", grid.best_params_)
print(f"Skor CV terbaik     : {grid.best_score_:.4f}")
print("Kombinasi dievaluasi:", len(grid.cv_results_["params"]))
print(f"Akurasi data uji    : {grid.score(X_test, y_test):.4f}")

**Penjelasan hasil:** terdapat $7 \times 2 \times 2 = 28$ kombinasi, masing-masing dievaluasi dengan 5-fold CV. Model terbaik dipilih berdasarkan rata-rata skor CV pada **data latih**, lalu dinilai sekali pada data uji yang belum pernah disentuh. Awalan `knn__` diperlukan karena parameter berada di dalam pipeline.

Pada dataset kecil seperti Iris, banyak kombinasi memberi skor yang sama atau hampir sama. Jika beberapa kombinasi seri, `GridSearchCV` memilih yang urutannya paling awal, jadi jangan menganggap parameter terpilih sebagai satu-satunya yang baik.

In [ ]:
res_df = pd.DataFrame(grid.cv_results_)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, w in zip(axes, ["uniform", "distance"]):
    sub = res_df[res_df["param_knn__weights"] == w]
    pv = sub.pivot_table(values="mean_test_score", index="param_knn__metric", columns="param_knn__n_neighbors")
    im = ax.imshow(pv.values, cmap="YlOrRd", vmin=res_df["mean_test_score"].min(), vmax=res_df["mean_test_score"].max(), aspect="auto")
    ax.set_xticks(range(pv.shape[1])); ax.set_xticklabels(pv.columns)
    ax.set_yticks(range(pv.shape[0])); ax.set_yticklabels(pv.index)
    for i in range(pv.shape[0]):
        for j in range(pv.shape[1]):
            ax.text(j, i, f"{pv.values[i, j]:.3f}", ha="center", va="center", fontsize=9)
    ax.set_title(f"weights = {w}"); ax.set_xlabel("n_neighbors"); ax.set_ylabel("metric")
fig.colorbar(im, ax=axes, label="Akurasi CV")
plt.show()

**Penjelasan hasil:** heatmap menampilkan skor CV tiap kombinasi (makin merah makin tinggi). Cari pola: apakah ada rentang $k$ yang konsisten baik, dan apakah salah satu metrik atau skema bobot unggul secara konsisten atau hanya selisih kecil. Pada Iris, daerah bagus biasanya cukup lebar, yang berarti model tidak sensitif terhadap pilihan $k$ yang wajar.

### 5.1 Metode Elbow untuk memilih $k$

Kita plot akurasi CV terhadap $k$ dan mencari titik ketika kenaikan akurasi mulai mendatar.

In [ ]:
k_range = range(1, 31)
means, stds = [], []
for k in k_range:
    p = Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=k))])
    s = cross_val_score(p, X_train, y_train, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=2024))
    means.append(s.mean()); stds.append(s.std())
means, stds = np.array(means), np.array(stds)

best_k = list(k_range)[means.argmax()]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_range), means, "bo-", ms=5)
ax.fill_between(list(k_range), means - stds, means + stds, alpha=0.15)
ax.axvline(best_k, color="red", ls="--", label=f"k terbaik = {best_k} ({means.max():.4f})")
ax.set_xlabel("k"); ax.set_ylabel("Akurasi CV"); ax.set_title("Elbow: akurasi vs k")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

print("Akurasi CV untuk beberapa k:")
for k in [1, 3, 5, 9, 15, 25, 30]:
    print(f"  k={k:<2}: {means[k-1]:.4f}")

**Penjelasan hasil:** garis merah menandai $k$ dengan rata-rata tertinggi, sedangkan area biru menunjukkan satu standar deviasi antar fold. Karena kurva biasanya datar di sekitar puncaknya, banyak $k$ yang secara statistik setara. Praktisnya pilih $k$ **yang lebih sederhana dan masuk akal di dalam area datar itu**, bukan sekadar maksimum yang bisa jadi kebetulan. Untuk $k$ yang sangat besar, akurasi mulai turun (*underfitting*) karena suara mayoritas makin kasar. Hindari $k$ genap pada klasifikasi biner karena bisa seri.

## 6. Evaluasi Kinerja KNN

Satu angka akurasi saja tidak cukup. Kita perlu tahu **di mana** model salah, **seberapa stabil** hasilnya, dan **bagaimana pengaruh ukuran data latih**.

### 6.1 Skor cross-validation

In [ ]:
best_model = grid.best_estimator_

cv_scores = cross_val_score(best_model, X_train, y_train, cv=5)
print("Skor tiap fold  :", cv_scores.round(4))
print(f"Rata-rata       : {cv_scores.mean():.4f}")
print(f"Standar deviasi : {cv_scores.std():.4f}")

**Penjelasan hasil:** lima skor menunjukkan kinerja pada lima pembagian data berbeda. Rata-rata mengukur kinerja umum, sedangkan standar deviasi mengukur stabilitas. Pada data latih hanya 120 sampel, satu kesalahan di fold berisi 24 sampel setara sekitar 4 poin persen, jadi variasi antar fold wajar.

### 6.2 Learning curve

Learning curve menunjukkan bagaimana skor latih dan skor validasi berubah seiring bertambahnya jumlah data latih. Berguna untuk mendiagnosis overfitting dan menilai apakah data tambahan akan membantu.

In [ ]:
from sklearn.model_selection import learning_curve

sizes, tr_sc, va_sc = learning_curve(
    best_model, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10), cv=5, shuffle=True, random_state=2024
)

plt.figure(figsize=(9, 5))
for sc, label, c in [(tr_sc, "Skor latih", "tab:blue"), (va_sc, "Skor validasi (CV)", "tab:orange")]:
    m, s = sc.mean(axis=1), sc.std(axis=1)
    plt.plot(sizes, m, "o-", label=label, color=c)
    plt.fill_between(sizes, m - s, m + s, alpha=0.15, color=c)
plt.xlabel("Jumlah data latih"); plt.ylabel("Akurasi")
plt.title("Learning curve"); plt.legend(loc="lower right"); plt.grid(alpha=0.3)
plt.show()

print("Ukuran data latih :", sizes)
print("Skor latih (mean) :", tr_sc.mean(axis=1).round(3))
print("Skor valid. (mean):", va_sc.mean(axis=1).round(3))

**Penjelasan hasil:** perhatikan bahwa kurva skor latih tampak datar di angka 1.0. Ini bukan kebetulan: model terbaik hasil grid search memakai `weights="distance"`, dan saat memprediksi titik latih, titik itu sendiri adalah tetangga dengan jarak 0 sehingga bobotnya mendominasi. Akibatnya skor latih selalu 1.0 dan tidak informatif untuk mendiagnosis overfitting pada konfigurasi ini. Kurva yang berguna di sini adalah **skor validasi**: dengan data latih sangat sedikit skornya rendah karena setiap kelas hanya punya segelintir contoh sehingga tetangga tidak representatif. Seiring bertambahnya data, skor validasi naik lalu mendatar. Jika kurva validasi sudah mendatar, menambah data tidak akan banyak membantu. Pada konfigurasi lain (misalnya `weights="uniform"`), skor latih tidak otomatis 1.0, sehingga celah antara kurva latih dan validasi dapat dipakai untuk menilai overfitting: celah lebar berarti model overfit dan butuh lebih banyak data atau $k$ yang lebih besar.

### 6.3 Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix (data uji)")
plt.show()

print(cm)
print(f"\nAkurasi data uji: {accuracy_score(y_test, y_pred):.4f}")

**Penjelasan hasil:** baris = label sebenarnya, kolom = label prediksi. Angka di diagonal adalah prediksi benar, angka di luar diagonal adalah kesalahan. Biasanya *setosa* terklasifikasi sempurna, dan sisa kesalahan, jika ada, berupa tertukarnya *versicolor* dengan *virginica*, sesuai dugaan dari scatter plot di awal. Data uji hanya 30 sampel, sehingga satu kesalahan saja menggeser akurasi sekitar 3.3 poin persen. Jangan menafsirkan perbedaan kecil sebagai perbedaan kualitas model yang nyata.

### 6.4 Classification report

In [ ]:
print(classification_report(y_test, y_pred, target_names=iris.target_names, digits=3))

# Verifikasi manual precision & recall untuk kelas 1 (versicolor)
tp = cm[1, 1]
fp = cm[:, 1].sum() - tp
fn = cm[1, :].sum() - tp
print(f"Versicolor -> TP={tp}, FP={fp}, FN={fn}")
print(f"Precision manual = TP/(TP+FP) = {tp/(tp+fp):.3f}")
print(f"Recall    manual = TP/(TP+FN) = {tp/(tp+fn):.3f}")

**Penjelasan hasil:** tiga metrik per kelas:

$$\text{Precision} = \frac{TP}{TP+FP} \qquad \text{Recall} = \frac{TP}{TP+FN} \qquad F1 = 2\cdot\frac{\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}$$

*Precision*: dari semua yang diprediksi kelas itu, berapa yang benar. *Recall*: dari semua yang sebenarnya kelas itu, berapa yang berhasil ditemukan. *F1* adalah rata-rata harmonik keduanya. Hitungan manual di bawah laporan cocok dengan angka di tabel. Kolom `support` adalah jumlah sampel sebenarnya per kelas.

## 7. KNN untuk Regresi

KNN juga bisa untuk regresi: prediksinya adalah **rata-rata** target dari $k$ tetangga. Pada contoh berikut kita sesuaikan kurva sinus berisik.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

rng = np.random.RandomState(0)
Xr = np.sort(5 * rng.rand(80, 1), axis=0)
yr = np.sin(Xr).ravel() + 0.2 * rng.randn(80)
Xg = np.linspace(0, 5, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, k in zip(axes, [1, 5, 25]):
    reg = KNeighborsRegressor(n_neighbors=k).fit(Xr, yr)
    ax.scatter(Xr, yr, c="gray", s=15, label="data")
    ax.plot(Xg, reg.predict(Xg), "r-", label=f"KNN k={k}")
    ax.plot(Xg, np.sin(Xg), "g--", alpha=0.6, label="sin(x) asli")
    ax.set_title(f"k = {k} | R2 latih = {reg.score(Xr, yr):.3f}")
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

**Penjelasan hasil:** garis merah adalah prediksi KNN, garis hijau putus-putus adalah fungsi sebenarnya. Dengan $k=1$ kurva mengikuti setiap titik berisik (overfitting, $R^2$ latih tinggi tetapi bergerigi). Dengan $k=25$ kurva terlalu halus sehingga meleset dari lengkung sinus (underfitting). Nilai di tengah memberikan kompromi terbaik. Prinsip bias-variance sama persis dengan klasifikasi.

## 8. Efisiensi: pilihan algoritma pencarian tetangga

Parameter `algorithm` menentukan cara mencari tetangga: `brute` (hitung semua jarak), `kd_tree`, `ball_tree`, atau `auto` (dipilih otomatis). Hasil prediksinya sama, hanya kecepatannya berbeda.

In [ ]:
import time

rng = np.random.RandomState(1)
Xbig = rng.rand(20000, 5)
ybig = (Xbig[:, 0] + Xbig[:, 1] > 1).astype(int)
Xq = rng.rand(2000, 5)

preds = {}
for algo in ["brute", "kd_tree", "ball_tree"]:
    m = KNeighborsClassifier(n_neighbors=5, algorithm=algo).fit(Xbig, ybig)
    t0 = time.time()
    preds[algo] = m.predict(Xq)
    print(f"{algo:<9}: {time.time() - t0:.3f} detik")

print("\nHasil prediksi sama untuk semua algoritma?",
      np.array_equal(preds["brute"], preds["kd_tree"]) and np.array_equal(preds["brute"], preds["ball_tree"]))

**Penjelasan hasil:** ketiga algoritma menghasilkan prediksi yang identik, tetapi waktunya berbeda. Pada dimensi rendah (di sini 5 fitur), struktur pohon (`kd_tree`, `ball_tree`) biasanya lebih cepat dari `brute`. Pada dimensi tinggi keunggulannya hilang dan `brute` bisa lebih baik, itulah mengapa `algorithm="auto"` (default) memilihkan secara otomatis. Waktu yang tepat bisa berbeda antar mesin.

## Ringkasan Akhir Chapter 4

| Konsep | Poin penting |
|:--|:--|
| **Metrik jarak** | Euclidean (garis lurus), Manhattan (grid), Minkowski (generalisasi), Chebyshev (selisih terbesar). Urutan: Manhattan $\geq$ Euclidean $\geq$ Chebyshev |
| **KNN** | Lazy learner: `fit` hanya menyimpan data, biaya ada di `predict` ($O(n \cdot d)$ per query) |
| **Scaling** | Wajib sebelum KNN karena fitur berskala besar mendominasi jarak |
| **Pilihan $k$** | $k$ kecil = varians tinggi (overfit), $k$ besar = bias tinggi (underfit). Tentukan lewat cross-validation |
| **Metrik jarak** | Tidak ada yang selalu terbaik; perlakukan sebagai hyperparameter |
| **Tuning** | `GridSearchCV` dengan scaler di dalam pipeline (`knn__n_neighbors`, dst.) |
| **Evaluasi** | CV (stabilitas), learning curve (efek ukuran data), confusion matrix dan classification report (letak kesalahan) |
| **Regresi** | `KNeighborsRegressor` merata-ratakan target tetangga |

**Tiga pelajaran utama:**

1. **Skalakan fitur** dan gunakan `Pipeline` agar bebas data leakage.
2. **Pilih $k$, metrik, dan bobot lewat validasi silang**, bukan tebakan. Waspadai kesimpulan dari data uji yang sangat kecil.
3. **Evaluasi lebih dari akurasi**: lihat confusion matrix, precision, recall, dan learning curve.